## This notebook will provide a demo walkthrough of BASIL.

I recommend you open this in Google Colab to use the function calculator.
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1Z9y8rDOoA5BRgmKdmsfgxnrkTqcAhn_r/view?usp=sharing)

## Hypothetical Chemical Reaction

Assuming we have an arbitrary chemical reaction, and we want to optimize (maximize) it's yield and purity. This can be framed as a multi-objective problem, so lets look at the inputs and outputs. Below, I have specified a searchspace for our process. We want to find the optimal values within it.

### Parameter Search space

| Variable | Symbol | Type | Range / Levels | Unit |
|---|---|---|---|---|
| Temperature | `T` | Continuous | [50, 90] | °C |
| pH | `p` | Continuous | [4.0, 8.0] | — |
| Catalyst amount | `c` | Discrete | {1, 2, 3, 4, 5} | level |
| Stirring speed | `s` | Discrete | {100, 200, 300, 400} | rpm |
| Catalyst type | `t` | Categorical | {A, B, C} | — |

\
We know that **yield** and **purity** are functions of the above parameters, but in reallife, we do not know what this function looks like. If we did, we could analytically solve for the maximum or minimum. Most reallife experiment and processes are blackbox, but we often want to optimize them (reduce cost, improve yield, decrease tocxicity). This is where Bayesian optimization can help.

\
So our hypothetical objectives are such that Yield and Purity are weighted sums of the input parameters (remember, we don't know this in reallife, and we want to maximize them):


$$\text{Yield}(T, p, c, s, t) = 0.30 \cdot S_T^Y + 0.25 \cdot S_p^Y + 0.20 \cdot S_c^Y + 0.15 \cdot S_s^Y + 0.10 \cdot S_t^Y$$


$$\text{Purity}(T, p, c, s, t) = 0.35 \cdot S_T^P + 0.25 \cdot S_p^P + 0.15 \cdot S_c^P + 0.15 \cdot S_s^P + 0.10 \cdot S_t^P$$





## Let's use BASIL to optimize the paramaters that maximize Yield and Purity.

### Create Workspace

Start by creating a workspace. Clicking the button would prompt you to create a folder in your computer.

<div align = "center">
<img src="imgs/work_space.jpg">
</div>

### Create Campaign

Then create an optimization campaign.

<div align = "center">
<img src="imgs/new_campaign.jpg">
</div>

### Campaign Information

Now we have to specify the type of campaign we want and the objectives to optimize. For this, we're doing a multi-objective optimization, and we'll use a Pareto strategy. You can leave the Surrogate Model and Acqusition function as defaults. A surrogate model builds a computational surrogate of our reallife process, while an acqusition function helps use to efficiently sample the searchspace.

We first one target and click "Add Another Target" to include the second one. Click Next!

<div align = "center">
<img src="imgs/setup_1.png">
</div>

<div align = "center">
<img src="imgs/setup_2.png">
</div>

Your newly created campaign will be in your workspace. Click on it!

<div align = "center">
<img src="imgs/campaign_in_workspace.png">
</div>

### Parameter Configuration

In the parameter configuration stage, we're going to specify the searchspace that we made above. The "Parm. Type" represents the type of variable we want to specify.

<div align = "center">
<img src="imgs/param_config_1.png">
</div>

Notice "Catalyst Amount" and "Stirring Speed", are *Discrete Numerical (Regular)*, and we can specify a step value for the variable. It is possible that we might have a list of discrete numbers that are not regularly spaced, in that case you'll use *Discrete Numerical (Irregular)* and type out the numbers.

<div align = "center">
<img src="imgs/param_config_2.png">
</div>

### Data Import

If you have historical data, you can import it. Since we do not have any, we'll skip this step by clicking next. The columns of the data you import must match the parameters and targets you set. You can generate a template of the ideal data columns by clicking *"Generate a template".*


<div align = "center">
<img src="imgs/historical_data_screen.png">
</div>

### Campaign Screen

Since we have not made any run, you'll see a screen like the one below.

<div align = "center">
<img src="imgs/run_screen_1.png">
</div>

You can see the Parameters and Targets you set by clicking the tab.

<div align = "center">
<img src="imgs/parameters_targets.png">
</div>

### Generate a Run

You can request a set of experiments by clicking the *Generate New Run* button. Specify the number of expriments you want to perform as well. The technical term for this number is called *batch size*. I requested 10 runs.


<div align = "center">
<img src="imgs/generate_run_1.png">
</div>

That should lead you to this screen below. When you perform the experiments, you can fill in the values for Yield and Purity. The first set of expriments are randomly sampled from the searchspace. This is for initialization.

<div align = "center">
<img src="imgs/run_screen_1.png">
</div>

In [ ]:
# @title Chemical Reaction Calculator: use this form below to get results for the recommended paramters. Assume that this is a simulated lab experiment. { display-mode: "form" }

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Parameter space ───────────────────────────────────────────────────────────
T_BOUNDS  = (50, 90)
P_BOUNDS  = (4.0, 8.0)
C_LEVELS  = [1, 2, 3, 4, 5]
S_LEVELS  = [100, 200, 300, 400]   # rpm
CAT_TYPES = ['A', 'B', 'C']

# Grid resolution for continuous variables
T_STEP = 0.5
P_STEP = 0.05

# ── Component shape: f(x) = coef*(x - optimum)^2 + 100 ───────────────────────
YIELD_PARAMS  = {'T': (-0.025,  80), 'p': (-6,   6.5), 'c': (-10,  4), 's': (-0.0015, 300)}
PURITY_PARAMS = {'T': (-0.030,  60), 'p': (-6,   5.5), 'c': (-10,  3), 's': (-0.0020, 200)}

# ── Objective weights ─────────────────────────────────────────────────────────
YIELD_WEIGHTS  = {'T': 0.30, 'p': 0.25, 'c': 0.20, 's': 0.15, 't': 0.10}
PURITY_WEIGHTS = {'T': 0.35, 'p': 0.25, 'c': 0.15, 's': 0.15, 't': 0.10}

CAT_YIELD  = {'A': 100, 'B': 70, 'C': 85}
CAT_PURITY = {'A': 70, 'B': 100, 'C': 85}

def score_component(x, coef, optimum):
    return coef * (x - optimum)**2 + 100

def chemical_reaction_objectives(T, p, c, s, t):
    yield_score = (
        YIELD_WEIGHTS['T'] * score_component(T, *YIELD_PARAMS['T']) +
        YIELD_WEIGHTS['p'] * score_component(p, *YIELD_PARAMS['p']) +
        YIELD_WEIGHTS['c'] * score_component(c, *YIELD_PARAMS['c']) +
        YIELD_WEIGHTS['s'] * score_component(s, *YIELD_PARAMS['s']) +
        YIELD_WEIGHTS['t'] * CAT_YIELD[t]
    )
    purity_score = (
        PURITY_WEIGHTS['T'] * score_component(T, *PURITY_PARAMS['T']) +
        PURITY_WEIGHTS['p'] * score_component(p, *PURITY_PARAMS['p']) +
        PURITY_WEIGHTS['c'] * score_component(c, *PURITY_PARAMS['c']) +
        PURITY_WEIGHTS['s'] * score_component(s, *PURITY_PARAMS['s']) +
        PURITY_WEIGHTS['t'] * CAT_PURITY[t]
    )
    return yield_score, purity_score

# Create input widgets using numeric text boxes instead of sliders
style = {'description_width': 'initial'}
t_input = widgets.BoundedFloatText(value=70.0, min=50.0, max=90.0, step=0.1, description='Temperature (°C):', style=style)
p_input = widgets.BoundedFloatText(value=6.0, min=4.0, max=8.0, step=0.01, description='pH:', style=style)
c_dropdown = widgets.Dropdown(options=C_LEVELS, value=3, description='Catalyst level:', style=style)
s_dropdown = widgets.Dropdown(options=S_LEVELS, value=200, description='Stirring (rpm):', style=style)
t_dropdown = widgets.Dropdown(options=CAT_TYPES, value='A', description='Catalyst type:', style=style)

calculate_button = widgets.Button(description='Calculate Results', button_style='primary')
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        y, pur = chemical_reaction_objectives(
            T=t_input.value,
            p=p_input.value,
            c=c_dropdown.value,
            s=s_dropdown.value,
            t=t_dropdown.value
        )
        print(f"--- Calculated Results ---")
        print(f"Yield:  {y:.2f}%")
        print(f"Purity: {pur:.2f}%")

calculate_button.on_click(on_button_clicked)

input_form = widgets.VBox([
    t_input, p_input, c_dropdown, s_dropdown, t_dropdown,
    calculate_button, output_area
])

display(input_form)

Fill in the values for all experiments.

<div align = "center">
<img src="imgs/run_1_with_results_1.png">
</div>

Click "Save Results" and then "Back to Runs"

<div align = "center">
<img src="imgs/run_1_with_results_2.png">
</div>

As you can see, our campaign screen now has a completed run!

<div align = "center">
<img src="imgs/campaign_screen_with_run.png">
</div>

We can perform a second batch of expriments by requesting new ones. Simply click the *Generate New Run* button. This time, I want to do 5 expriments.

<div align = "center">
<img src="imgs/generate_run_2.png">
</div>

You'll see this screen while BASIL is crunching numbers. Should take less that a minute for this demo.

<div align = "center">
<img src="imgs/progress_screen.png">
</div>

Now we repeat the same thing we did for run one. Do the experiments and fill in the values.

<div align = "center">
<img src="imgs/run_screen_2.png">
</div>

You can see that we're already converging to very high yields and purity! Awesome Numbers!

<div align = "center">
<img src="imgs/run_2_with_results_1.png">
</div>

You can do more expriments if you like. Although the point of Bayesian optimization is to minimize the number of experiments you do. Since this is a demo, I did 4 run and a total of 25 expriments across all of them.

<div align = "center">
<img src="imgs/all_runs.png">
</div>

### Visualizations and Explanations

You can use BASIL to make plots of your expriment/process.


<div align = "center">
<img src="imgs/plot_1.png">
</div>

<div align = "center">
<img src="imgs/plot_2.png">
</div>


You can also create [SHAP plots](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/An%20introduction%20to%20explainable%20AI%20with%20Shapley%20values.html) to help you explain how the inputs affect the objectives.

<div align = "center">
<img src="imgs/explanation_1.png">
</div>


### Optimized Objectives

Since the function we optimized was hypothetically derived, I am showing you the optimial values here. Check BASIL to see how may iterations it took to find something similar. Note that Yield and Purity are competing objectives, so getting a high value for one, sometimes reduces the other. This is called the Pareto frontier, and what you got were a set of tradeoffs.

Each parameter contributes via: $S_i(x) = -a_i(x - x_{opt})^2 + 100$

| Parameter | Yield Optimum | Purity Optimum |
|-----------|---------------|----------------|
| Temperature (T) | 80°C | 60°C |
| pH (p) | 6.5 | 5.5 |
| Catalyst amount (c) | 4 | 3 |
| Stirring speed (s) | 300 rpm | 200 rpm |

**Catalyst type scores:**

| Type | Yield ($S_t^Y$) | Purity ($S_t^P$) |
|------|---------|----------|
| A | 100 | 70 |
| B | 70 | 100 |
| C | 85 | 85 |

**Key insight:** Temperature and catalyst type create the primary trade-off between objectives.

 Congratulations! You've successfully optimized a process with BASIL. Of course, a reallife process might not be as easy, but now you have the right tools to optimize it in a few experiments.